# Etapa 3.1 - MapReduce con Spark RDDs

## Configuración del entorno Spark

In [1]:
!python3 -m pip install pyspark



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip


In [2]:
from pyspark import SparkContext, SparkConf

conf = SparkConf().setAppName("TP_IBD_MapReduce").setMaster("local[*]")
sc = SparkContext.getOrCreate(conf=conf)
sc.setLogLevel("WARN")
print("SparkContext activo:", sc.version)

The operation couldn’t be completed. Unable to locate a Java Runtime.
Please visit http://www.java.com for information on installing Java.

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pyspark/bin/spark-class: line 97: CMD: bad array subscript
head: illegal line count -- -1


PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

## Carga de los CSV y utilidades de parseo

Leemos cada CSV como RDD de texto; con el encabezado armamos un mapa `columna -> indice`. Como `precio_total`/`subtotal` ya no son columnas, reconstruimos el monto por venta desde el detalle (`SUM(cantidad*precio_unidad)`).

In [ ]:
DATA_DIR = "data"


def cargar_csv(nombre_archivo):
    # Devuelve (rdd_de_filas, {columna: indice}) leyendo el CSV con encabezado
    rdd_texto = sc.textFile(f"{DATA_DIR}/{nombre_archivo}")
    header = rdd_texto.first()
    col = {nombre: i for i, nombre in enumerate(header.split(","))}
    rdd_filas = (
        rdd_texto
        .filter(lambda linea: linea != header)
        .map(lambda linea: linea.split(","))
    )
    return rdd_filas, col


ventas_rdd, V = cargar_csv("ventas.csv")
detalle_ventas_rdd, DV = cargar_csv("detalle_ventas.csv")
ventas_rdd.cache()
detalle_ventas_rdd.cache()

# metodo_pago es una entidad: catalogo id -> nombre
metodos_rdd, M = cargar_csv("metodos_pago.csv")
METODO_NOMBRE = {f[M["metodo_pago_id"]]: f[M["nombre"]] for f in metodos_rdd.collect()}

# Monto por venta = SUM(cantidad * precio_unidad) desde el detalle
monto_por_venta = (
    detalle_ventas_rdd
    .map(lambda f: (f[DV["venta_id"]],
                    float(f[DV["cantidad"]]) * float(f[DV["precio_unidad"]])))
    .reduceByKey(lambda a, b: a + b)
)
monto_por_venta.cache()

print("Columnas VENTAS:", V)
print("Columnas DETALLE_VENTAS:", DV)
print("Metodos de pago:", METODO_NOMBRE)
print("Ventas con monto reconstruido:", monto_por_venta.count())

---
## Consulta 1 - Facturacion, cantidad de ventas y ticket promedio por sucursal

Map: unir `monto_por_venta` con la sucursal de cada venta -> `(puntos_de_venta_id, (monto, 1))`. Reduce: `reduceByKey` suma componente a componente y un `mapValues` deriva el ticket promedio.

In [ ]:
# Map: (puntos_de_venta_id, (monto, 1))
venta_sucursal = ventas_rdd.map(
    lambda f: (f[V["venta_id"]], f[V["puntos_de_venta_id"]])
)
map_c1 = (
    monto_por_venta.join(venta_sucursal)
    .map(lambda kv: (kv[1][1], (kv[1][0], 1)))
)

# Reduce: (suma_monto, cantidad_ventas) por sucursal
reduce_c1 = map_c1.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

# Ticket promedio = facturacion / cantidad
resultado_c1 = reduce_c1.mapValues(
    lambda v: (round(v[0], 2), v[1], round(v[0] / v[1], 2))
)

print("Sucursal | Facturación total | Cant. ventas | Ticket promedio")
print("-" * 65)
for pv_id, (fact, cant, ticket) in sorted(resultado_c1.collect(), key=lambda x: -x[1][0]):
    print(f"  {pv_id:>3}    | {fact:>16,.2f} | {cant:>11} | {ticket:>14,.2f}")

---
## Consulta 2 - Top 10 productos por profit

Map: por cada linea, `(product_id, cantidad*(precio_unidad - costo_unidad))`. Reduce: `reduceByKey` suma el profit y `takeOrdered(10)` trae el top de forma distribuida.

In [ ]:
# Map: (product_id, cantidad * (precio_unidad - costo_unidad))
map_c2 = detalle_ventas_rdd.map(
    lambda f: (
        f[DV["product_id"]],
        float(f[DV["cantidad"]]) * (float(f[DV["precio_unidad"]]) - float(f[DV["costo_unidad"]]))
    )
)

# Reduce: profit total por producto
reduce_c2 = map_c2.reduceByKey(lambda a, b: a + b)

# Top 10 distribuido
top10_c2 = reduce_c2.takeOrdered(10, key=lambda x: -x[1])

print("Top 10 productos por profit total")
print("Pos | product_id | Profit total")
print("-" * 40)
for pos, (pid, profit) in enumerate(top10_c2, start=1):
    print(f" {pos:>2} | {pid:>10} | {profit:>14,.2f}")

---
## Consulta 3 - Distribucion de ventas e ingresos por metodo de pago

Map: unir `monto_por_venta` con `metodo_pago_id` -> `(metodo_pago_id, (1, monto))`. Reduce: `reduceByKey` suma cantidad y monto por metodo; el nombre se resuelve con el catalogo al imprimir.

In [ ]:
# Map: (metodo_pago_id, (1, monto))
venta_metodo = ventas_rdd.map(
    lambda f: (f[V["venta_id"]], f[V["metodo_pago_id"]])
)
map_c3 = (
    monto_por_venta.join(venta_metodo)
    .map(lambda kv: (kv[1][1], (1, kv[1][0])))
)

# Reduce: (cantidad_ventas, monto_total) por metodo
reduce_c3 = map_c3.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
resultado_c3 = reduce_c3.collect()

total_ventas = sum(cant for _, (cant, _) in resultado_c3)

print("Metodo de pago    | Cant. ventas |   % | Monto total")
print("-" * 60)
for metodo_id, (cant, monto) in sorted(resultado_c3, key=lambda x: -x[1][0]):
    nombre = METODO_NOMBRE.get(metodo_id, metodo_id)
    pct = 100.0 * cant / total_ventas
    print(f"{nombre:<17} | {cant:>11} | {pct:>4.1f} | {monto:>14,.2f}")

---
## Cierre

In [ ]:
ventas_rdd.unpersist()
detalle_ventas_rdd.unpersist()
monto_por_venta.unpersist()
sc.stop()
print("SparkContext detenido.")